# Dust attenuation vs. bulge-to-total ratio in quiescent galaxies at $z\sim0.3$

**Question.** Among *quenched* galaxies at $z\simeq0.3$ with $\log_{10}M_\star>10.25$, does the dust
attenuation $A_V$ trend with morphology (stellar bulge-to-total ratio $B/T$), and does that trend
depend on how much molecular gas is left ($M_{\rm H_2}/M_\star$)?

**Method.**
* Single epoch — snapshot **134** of `cis100` ($z=0.305$); no progenitor tracking is needed.
* **Attenuation** from the CAESAR/pyloser photometry stored *in the catalog*: the dust-attenuated
  and dust-free (intrinsic) $V$-band magnitudes give $A_V = m_V^{\rm dust}-m_V^{\rm nodust}$
  (equivalently $-2.5\log_{10}(F^{\rm dust}_V/F^{\rm nodust}_V)$). *No Powderday run is involved.*
* **Morphology** from CAESAR's kinematic decomposition `rotation.stellar_BoverT` (with
  `rotation.stellar_kappa_rot` as a cross-check axis). This is a kinematic $B/T$, **not** a Sérsic fit.
* **H$_2$ fraction** $=M_{\rm H_2}/M_\star$ (`masses.H2/masses.stellar`), the same definition used in
  `quench_mode_vs_sigma_gas.ipynb`. Binning in H$_2$ controls for the dust–gas correlation so the
  residual $A_V$–$B/T$ trend isolates *geometry* rather than *gas budget*.

> **One thing to confirm on the cluster first (Part 1):** the repo has never read pyloser photometry,
> so the exact HDF5 key names for the dust / no-dust $V$-band are unknown here. Part 1 discovers them;
> set `AV_KEY_DUST` / `AV_KEY_NODUST` in Part 1b to match before running the science cells.

# Part 0 — Setup & configuration

In [ ]:
import os
import re
import numpy as np
import h5py
import matplotlib.pyplot as plt

from astropy.cosmology import Planck15 as COSMO   # matches the repo's quenching batch
from scipy.stats import spearmanr, pearsonr

from simbanator.io.simba import Simulation

# ── simulation / epoch ──
SIM_NAME = "cis100"
SNAP_Z03 = 134                 # z = 0.305 in the box-100 snap→z map (closest to z~0.3)

# ── sample selection ──
LOGM_MIN       = 10.25         # log10(M*/Msun) floor  (user spec)
PASSIVE_FACTOR = 0.2           # quiescent if sSFR < PASSIVE_FACTOR / t_H(z)  (== QT threshold)
NSTAR_MIN      = 20            # min star particles — B/T (kinematic decomp.) needs resolved stars
NGAS_MIN       = 0             # do NOT require gas: gas-poor quenched galaxies are the low-H2 bin

# ── H2 gas-fraction bins (M_H2 / M*, in %). Quenched galaxies are gas-poor → fine low-frac bins. ──
H2FRAC_BINS_PCT = [(0.0, 0.5, "<0.5%",  "C0"),
                   (0.5, 2.0, "0.5–2%", "C2"),
                   (2.0, 5.0, "2–5%",   "C1"),
                   (5.0, 1e9, ">5%",    "C3")]

sim = Simulation(SIM_NAME)
Z_EPOCH = float(sim.get_z_from_snap(SNAP_Z03))
T_H_YR  = COSMO.age(Z_EPOCH).value * 1e9
SSFR_Q  = PASSIVE_FACTOR / T_H_YR                      # quiescent threshold [1/yr]

OUT     = os.path.join(os.getcwd(), "output", SIM_NAME)
PLOTDIR = os.path.join(OUT, "plots", "attenuation_bt_z03")
TABLEDIR = os.path.join(OUT, "tables")
os.makedirs(PLOTDIR, exist_ok=True)
os.makedirs(TABLEDIR, exist_ok=True)
def figpath(name): return os.path.join(PLOTDIR, name)

print(f"simulation : {SIM_NAME}")
print(f"epoch      : snap {SNAP_Z03}  ->  z = {Z_EPOCH:.3f}   t_H = {T_H_YR/1e9:.3f} Gyr")
print(f"selection  : quiescent (sSFR < {SSFR_Q:.2e}/yr) + log10 M* > {LOGM_MIN} + nstar >= {NSTAR_MIN}")

In [ ]:
# ── global plot style (verbatim from residual_dust_quenching.ipynb §0b / quench_mode §0b) ──
import functools
import matplotlib as mpl
from matplotlib.axes import Axes
from matplotlib.figure import Figure

PLOT_FIG_SCALE, PLOT_FONT_BASE, PLOT_FONT_FLOOR = 1.45, 15, 14
mpl.rcParams.update({
    "font.size": PLOT_FONT_BASE, "axes.titlesize": PLOT_FONT_BASE + 2,
    "axes.labelsize": PLOT_FONT_BASE + 1, "xtick.labelsize": PLOT_FONT_BASE,
    "ytick.labelsize": PLOT_FONT_BASE, "legend.fontsize": PLOT_FONT_BASE,
    "figure.titlesize": PLOT_FONT_BASE + 4, "axes.titleweight": "bold",
    "figure.dpi": 110, "lines.linewidth": 2.0,
})
def _bump(fs):
    if isinstance(fs, (int, float)) and not isinstance(fs, bool) and fs < PLOT_FONT_FLOOR:
        return PLOT_FONT_FLOOR
    return fs
def _patch_fontsize(cls, names):
    for nm in names:
        orig = getattr(cls, nm, None)
        if orig is None or getattr(orig, "_fontfloor", False):
            continue
        @functools.wraps(orig)
        def wrapper(*a, __orig=orig, **kw):
            if "fontsize" in kw: kw["fontsize"] = _bump(kw["fontsize"])
            if "size" in kw:     kw["size"]     = _bump(kw["size"])
            return __orig(*a, **kw)
        wrapper._fontfloor = True
        setattr(cls, nm, wrapper)
_patch_fontsize(Axes, ["set_title", "set_xlabel", "set_ylabel", "legend", "text", "annotate"])
_patch_fontsize(Figure, ["suptitle", "legend"])
if not getattr(plt.figure, "_figscale", False):
    _orig_figure = plt.figure
    @functools.wraps(_orig_figure)
    def _figure(*a, **kw):
        base = kw.get("figsize") or mpl.rcParams["figure.figsize"]
        kw["figsize"] = (base[0] * PLOT_FIG_SCALE, base[1] * PLOT_FIG_SCALE)
        return _orig_figure(*a, **kw)
    _figure._figscale = True
    plt.figure = _figure
print(f"plot style: fonts>={PLOT_FONT_FLOOR}pt, figures x{PLOT_FIG_SCALE}")

# Part 1 — Load the catalog & **discover** the pyloser photometry keys

The catalog is loaded once. The discovery cell enumerates every photometry-like entry both on the
live CAESAR galaxy object (`g.absmag`, `g.absmag_nodust`, …) **and** in the HDF5 dict group
(`galaxy_data/dicts/absmag.*`, …), since the exact layout depends on the pyloser/CAESAR version.

**Read the output and pick the dust and no-dust $V$-band keys**, then set them in Part 1b.

In [ ]:
cs      = sim.load_catalog(snap=SNAP_Z03)
CATFILE = sim.get_caesar_file(SNAP_Z03)
NGAL    = len(cs.galaxies)
print(f"loaded {NGAL} galaxies from  {CATFILE}")

_PHOT_RE = re.compile(r"mag|phot|lum|flux|absorp|dust|nodust|band", re.I)

# ── (a) live-object photometry attributes ──
print("\n── live CAESAR galaxy object ──")
g0 = cs.galaxies[0]
_obj_phot = {}
for attr in ("absmag", "absmag_nodust", "appmag", "appmag_nodust", "photometry", "band_mags"):
    val = getattr(g0, attr, None)
    if isinstance(val, dict):
        _obj_phot[attr] = sorted(val.keys())
        print(f"  g.{attr:16s} -> {len(val)} bands: {list(val)[:12]}{' …' if len(val)>12 else ''}")
if not _obj_phot:
    extra = sorted(k for k in vars(g0) if _PHOT_RE.search(k)) if hasattr(g0, "__dict__") else []
    print("  (no absmag/appmag dict attrs found)  other phot-like attrs:", extra or "none")

# ── (b) HDF5 dict group ──
print("\n── HDF5 galaxy_data/dicts (photometry-like keys) ──")
_hdf_phot = []
with h5py.File(CATFILE, "r") as f:
    dd = f.get("galaxy_data/dicts")
    if dd is not None:
        _hdf_phot = sorted(k for k in dd.keys() if _PHOT_RE.search(k))
        for k in _hdf_phot:
            print(f"  galaxy_data/dicts/{k:28s} shape={dd[k].shape}")
        if not _hdf_phot:
            print("  (none)  all dict keys:", sorted(dd.keys())[:40])
    else:
        print("  no galaxy_data/dicts group")
print("\nSet AV_KEY_DUST / AV_KEY_NODUST in Part 1b from the V-band entries above.")

# Part 1b — Photometry accessor configuration  *(edit to match Part 1 output)*

Fill in the two $V$-band keys you saw above. Defaults are best-guesses for a CAESAR/pyloser catalog
(`absmag.v` / `absmag_nodust.v`); **confirm them** — band labels vary (`v`, `V`, `johnson_v`,
`Johnson_V`), and photometry may sit on the live object instead of the HDF5 dicts.

* `PHOT_SOURCE` — `"hdf5"` (read the aligned `galaxy_data/dicts/<key>` array) or `"object"`
  (read `getattr(g, ATTR)[BAND]` per galaxy).
* `PHOT_IS_MAG` — `True` if the values are magnitudes ($A_V=m^{\rm dust}-m^{\rm nodust}$),
  `False` if they are fluxes/luminosities ($A_V=-2.5\log_{10}(F^{\rm dust}/F^{\rm nodust})$).

In [ ]:
# ============================ EDIT THESE ============================
PHOT_SOURCE = "hdf5"          # "hdf5"  ->  keys below are galaxy_data/dicts/<key>
                              # "object"->  keys below are (attr, band): e.g. ("absmag", "v")
PHOT_IS_MAG = True            # True: magnitudes; False: fluxes/luminosities

# --- if PHOT_SOURCE == "hdf5" ---
AV_KEY_DUST   = "absmag.v"          # dust-attenuated V
AV_KEY_NODUST = "absmag_nodust.v"   # intrinsic (dust-free) V

# --- if PHOT_SOURCE == "object" ---
AV_ATTR_DUST   = ("absmag",        "v")
AV_ATTR_NODUST = ("absmag_nodust", "v")
# ===================================================================

def _av_from_pair(mdust, mnodust):
    """A_V from a dust / no-dust pair (magnitudes or fluxes). Positive = attenuated."""
    mdust   = np.asarray(mdust, float)
    mnodust = np.asarray(mnodust, float)
    with np.errstate(all="ignore"):
        if PHOT_IS_MAG:
            av = mdust - mnodust
        else:
            av = -2.5 * np.log10(mdust / mnodust)
    av[~np.isfinite(av)] = np.nan
    return av

def compute_AV():
    """Per-galaxy A_V aligned to cs.galaxies order (length NGAL). NaN where photometry is absent."""
    if PHOT_SOURCE == "hdf5":
        with h5py.File(CATFILE, "r") as f:
            dd = f["galaxy_data/dicts"]
            for k in (AV_KEY_DUST, AV_KEY_NODUST):
                if k not in dd:
                    raise KeyError(f"'{k}' not in galaxy_data/dicts — check Part 1 discovery output. "
                                   f"available: {sorted(dd.keys())[:30]}")
            md_ = np.asarray(dd[AV_KEY_DUST][:], float)
            mn_ = np.asarray(dd[AV_KEY_NODUST][:], float)
        if len(md_) != NGAL:
            raise ValueError(f"photometry length {len(md_)} != n_galaxies {NGAL} (alignment broken)")
        return _av_from_pair(md_, mn_)
    elif PHOT_SOURCE == "object":
        (ad, bd), (an, bn) = AV_ATTR_DUST, AV_ATTR_NODUST
        md_ = np.array([float(getattr(g, ad, {}).get(bd, np.nan)) for g in cs.galaxies])
        mn_ = np.array([float(getattr(g, an, {}).get(bn, np.nan)) for g in cs.galaxies])
        return _av_from_pair(md_, mn_)
    raise ValueError(f"unknown PHOT_SOURCE={PHOT_SOURCE!r}")

AV = compute_AV()
_nok = int(np.isfinite(AV).sum())
print(f"A_V computed for {_nok}/{NGAL} galaxies "
      f"({'magnitudes' if PHOT_IS_MAG else 'fluxes'}, source={PHOT_SOURCE})")
if _nok:
    print(f"  A_V range: {np.nanpercentile(AV,1):.2f} … {np.nanmedian(AV):.2f} (med) … "
          f"{np.nanpercentile(AV,99):.2f}")
else:
    print("  [!] no finite A_V — the keys in Part 1b do not match the catalog; fix and re-run.")

# Part 2 — Build the galaxy table & apply the selection

Everything below reads the **live CAESAR object** (proven access path in this repo): `g.masses`,
`g.sfr`, `g.rotation`, `g.ngas`, `g.nstar`. The selection is quiescent **and** massive **and**
star-resolved; H$_2$-poor galaxies are kept (they populate the lowest H$_2$-fraction bin).

In [ ]:
def _mass(g, k):     return float(g.masses.get(k, np.nan))
def _rot(g, k):      return float(g.rotation.get(k, np.nan)) if hasattr(g, "rotation") else np.nan

mstar = np.array([_mass(g, "stellar") for g in cs.galaxies])
mh2   = np.array([_mass(g, "H2")      for g in cs.galaxies])
mgas  = np.array([_mass(g, "gas")     for g in cs.galaxies])
sfr   = np.array([float(g.sfr)        for g in cs.galaxies])
ngas  = np.array([int(g.ngas)         for g in cs.galaxies])
nstar = np.array([int(g.nstar)        for g in cs.galaxies])
BT    = np.array([_rot(g, "stellar_BoverT")   for g in cs.galaxies])   # kinematic bulge/total
kappa = np.array([_rot(g, "stellar_kappa_rot") for g in cs.galaxies])  # disciness (cross-check axis)

with np.errstate(all="ignore"):
    logM  = np.log10(np.where(mstar > 0, mstar, np.nan))
    ssfr  = np.where(mstar > 0, sfr / mstar, np.nan)
    fH2   = np.where(mstar > 0, mh2 / mstar, np.nan)      # M_H2 / M*   (fraction)
    h2pct = 100.0 * fH2

# ── selection mask ──
sel = (
    (logM > LOGM_MIN)
    & (ssfr < SSFR_Q)              # quiescent
    & (nstar >= NSTAR_MIN)
    & (ngas >= NGAS_MIN)
    & np.isfinite(BT)             # need a morphology
)
NSEL = int(sel.sum())
print(f"selected {NSEL} / {NGAL} galaxies")
print(f"  with finite A_V     : {int((sel & np.isfinite(AV)).sum())}")
print(f"  with finite B/T     : {int((sel & np.isfinite(BT)).sum())}")
print(f"  H2 fraction (of sel): median {np.nanmedian(h2pct[sel]):.2f}%, "
      f"[16–84] {np.nanpercentile(h2pct[sel],16):.2f}–{np.nanpercentile(h2pct[sel],84):.2f}%, "
      f"H2=0 for {int((sel & (mh2 <= 0)).sum())} galaxies")
print(f"  B/T  (of sel)       : median {np.nanmedian(BT[sel]):.2f}, "
      f"[16–84] {np.nanpercentile(BT[sel],16):.2f}–{np.nanpercentile(BT[sel],84):.2f}")

# Part 3 — Assign H$_2$-fraction bins & report composition

In [ ]:
def h2_bin_index(pct):
    """Index into H2FRAC_BINS_PCT (or -1). NaN H2 fraction -> lowest bin (treated as gas-poor)."""
    if not np.isfinite(pct):
        pct = 0.0
    for i, (lo, hi, _, _) in enumerate(H2FRAC_BINS_PCT):
        if lo <= pct < hi:
            return i
    return -1

h2idx = np.array([h2_bin_index(p) for p in h2pct])

print(f"{'H2 bin':>10s} {'N(sel)':>8s} {'N(A_V)':>8s} {'med A_V':>9s} {'med B/T':>9s}")
for i, (lo, hi, lab, _) in enumerate(H2FRAC_BINS_PCT):
    m  = sel & (h2idx == i)
    ma = m & np.isfinite(AV)
    print(f"{lab:>10s} {int(m.sum()):8d} {int(ma.sum()):8d} "
          f"{np.nanmedian(AV[ma]) if ma.any() else np.nan:9.2f} "
          f"{np.nanmedian(BT[m]) if m.any() else np.nan:9.2f}")

# Part 4 — Headline figure: $A_V$ vs. $B/T$, by H$_2$ gas fraction

For each H$_2$ bin, the median $A_V$ is tracked across bins of $B/T$ (shaded 16–84% band); the raw
galaxies are shown as a faint scatter. The Spearman $\rho$ (and $p$) of $A_V$ vs $B/T$ *within* each
H$_2$ bin is printed in the legend — that is the controlled test of "does geometry matter at fixed
gas budget?".

In [ ]:
BT_EDGES    = np.linspace(0.0, 1.0, 9)     # 8 bins in B/T ∈ [0,1]
MIN_PER_BIN = 5                             # min galaxies for a median in a B/T cell
BAND_PCT    = (16, 84)

def _med_band(x, y, edges, min_n=MIN_PER_BIN, pcts=BAND_PCT):
    cen = 0.5 * (edges[:-1] + edges[1:])
    med = np.full(len(cen), np.nan); lo = np.full(len(cen), np.nan); hi = np.full(len(cen), np.nan)
    for b in range(len(cen)):
        v = y[(x >= edges[b]) & (x < edges[b + 1]) & np.isfinite(y)]
        if len(v) >= min_n:
            med[b] = np.median(v); lo[b] = np.percentile(v, pcts[0]); hi[b] = np.percentile(v, pcts[1])
    return cen, med, lo, hi

def plot_av_vs_bt(xkey="BT", xarr=None, xlabel=r"stellar $B/T$", fname="attenuation_vs_bt_byH2.png"):
    X = BT if xarr is None else xarr
    base = sel & np.isfinite(AV) & np.isfinite(X)
    if not base.any():
        print("no galaxies with finite A_V and", xkey, "— check Part 1b keys."); return
    fig, ax = plt.subplots(figsize=(7.2, 5.6))
    edges = BT_EDGES if xkey == "BT" else np.linspace(np.nanpercentile(X[base],1),
                                                      np.nanpercentile(X[base],99), 9)
    for i, (lo_, hi_, lab, col) in enumerate(H2FRAC_BINS_PCT):
        m = base & (h2idx == i)
        if m.sum() < MIN_PER_BIN:
            continue
        ax.scatter(X[m], AV[m], s=10, color=col, alpha=0.18, lw=0)
        cen, med, blo, bhi = _med_band(X[m], AV[m], edges)
        rho, p = spearmanr(X[m], AV[m])
        ax.plot(cen, med, "-o", color=col, ms=4, lw=2.0,
                label=fr"{lab}  (n={int(m.sum())}, $\rho$={rho:+.2f}, p={p:.1g})")
        ax.fill_between(cen, blo, bhi, color=col, alpha=0.12, lw=0)
    ax.axhline(0.0, ls=":", c="0.5", lw=1)
    if xkey == "BT":
        ax.axvline(0.5, ls="--", c="0.6", lw=1)   # nominal disc/bulge divide
    ax.set_xlabel(xlabel); ax.set_ylabel(r"$A_V = m_V^{\rm dust}-m_V^{\rm nodust}$  [mag]")
    ax.set_title(fr"Dust attenuation vs. morphology — quenched, $\log M_\star>{LOGM_MIN}$, $z={Z_EPOCH:.2f}$")
    ax.legend(title=r"H$_2$ gas fraction $M_{\rm H_2}/M_\star$", fontsize=11, loc="best")
    fig.tight_layout(); fig.savefig(figpath(fname), dpi=130, bbox_inches="tight"); plt.show()
    print("saved ->", figpath(fname))

plot_av_vs_bt()

# Part 5 — Robustness

1. **Cross-check morphology axis.** Repeat against $\kappa_{\rm rot}^\star$ (disciness) — a high-$B/T$
   trend should appear as a *low*-$\kappa$ trend.
2. **Correlations & partial correlation.** Spearman $A_V$–$B/T$ overall and within each H$_2$ bin; a
   simple stratified (Fisher-$z$–pooled) partial correlation controlling for H$_2$ fraction, since
   $A_V$ and H$_2$ are themselves correlated (dust tracks gas).
3. **Table dump** of the per-galaxy sample for downstream use.

In [ ]:
# 1 — same figure against kappa_rot (disciness)
plot_av_vs_bt(xkey="kappa", xarr=kappa, xlabel=r"stellar $\kappa_{\rm rot}$",
              fname="attenuation_vs_kappa_byH2.png")

In [ ]:
# 2 — correlations
base = sel & np.isfinite(AV) & np.isfinite(BT)
rho_all, p_all = spearmanr(BT[base], AV[base])
print(f"overall  Spearman A_V vs B/T : rho={rho_all:+.3f}  p={p_all:.2g}  (n={int(base.sum())})")
print(f"overall  Spearman A_V vs H2% : "
      f"{spearmanr(h2pct[base], AV[base])[0]:+.3f}  "
      f"p={spearmanr(h2pct[base], AV[base])[1]:.2g}")

print("\nwithin H2 bins (controls for gas budget):")
_zs, _ws = [], []   # Fisher-z pooling for a crude partial correlation
for i, (lo_, hi_, lab, _) in enumerate(H2FRAC_BINS_PCT):
    m = base & (h2idx == i)
    if m.sum() < 5:
        print(f"  {lab:>8s}: n={int(m.sum())} (too few)"); continue
    rho, p = spearmanr(BT[m], AV[m]); n = int(m.sum())
    print(f"  {lab:>8s}: rho={rho:+.3f}  p={p:.2g}  n={n}")
    if abs(rho) < 0.999 and n > 3:
        _zs.append(np.arctanh(rho)); _ws.append(n - 3)
if _zs:
    z_pool = np.average(_zs, weights=_ws)
    print(f"\nH2-stratified partial Spearman A_V–B/T (Fisher-z pooled): rho_partial = {np.tanh(z_pool):+.3f}")

In [ ]:
# 3 — dump the sample table (finite-A_V galaxies)
_cols = dict(gid=np.array([int(g.GroupID) for g in cs.galaxies]),
             logMstar=logM, ssfr=ssfr, h2frac_pct=h2pct, BoverT=BT, kappa=kappa, A_V=AV,
             h2bin=h2idx)
_tab = sel & np.isfinite(AV)
outp = os.path.join(TABLEDIR, f"attenuation_bt_z03_snap{SNAP_Z03}.hdf5")
with h5py.File(outp, "w") as o:
    o.attrs.update(dict(sim=SIM_NAME, snap=SNAP_Z03, z=Z_EPOCH, logM_min=LOGM_MIN,
                        ssfr_q=SSFR_Q, phot_source=PHOT_SOURCE, phot_is_mag=int(PHOT_IS_MAG),
                        av_key_dust=str(AV_KEY_DUST), av_key_nodust=str(AV_KEY_NODUST)))
    for k, v in _cols.items():
        o.create_dataset(k, data=np.asarray(v)[_tab])
print(f"wrote {int(_tab.sum())} galaxies -> {outp}")

# Interpretation *(fill after the cluster run)*

* If the $A_V$–$B/T$ slope **survives** at fixed H$_2$ fraction (nonzero partial $\rho$), dust
  attenuation in quenched galaxies is set by *geometry* (compact bulge-dominated sightlines) beyond
  the raw dust/gas budget.
* If it **collapses** once H$_2$ is fixed, the apparent $A_V$–morphology trend is really just the
  H$_2$–morphology correlation (gas-rich galaxies are diskier *and* dustier).
* Compare with `quench_mode_vs_sigma_gas.ipynb` §7·1, where $M_{\rm dust}/M_\star$ (a *budget* proxy,
  not a sightline attenuation) was tracked vs $\Delta$MS by the same H$_2$ bins.